# Clase 142 — Mecanismos de atención

La **atención** destrabó el NLP moderno: en vez de un único vector de contexto, el
modelo pondera **todos** los estados relevantes. El corazón es la **scaled
dot-product attention**: `softmax(QK^T / √d) · V`.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. Scaled dot-product attention a mano en numpy

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def atencion(Q, K, V):
    d = Q.shape[-1]
    pesos = softmax(Q @ K.T / np.sqrt(d))   # softmax(QK^T / sqrt(d))
    return pesos @ V, pesos

seq_q, seq_k, d = 3, 4, 8
Q = rng.standard_normal((seq_q, d))
K = rng.standard_normal((seq_k, d))
V = rng.standard_normal((seq_k, d))
salida, pesos = atencion(Q, K, V)
print("salida:", salida.shape, "| pesos:", pesos.shape)

## 2. Por qué escalar por `√d`

In [ ]:
print("cada fila de pesos suma 1:", np.allclose(pesos.sum(axis=1), 1.0))
print("pesos[0]:", np.round(pesos[0], 3))

# Sin escalar, QK^T crece con d y el softmax se satura (una entropía menor).
sin_escala = softmax(Q @ K.T)
ent = lambda p: float(-(p * np.log(p + 1e-9)).sum(1).mean())
print("entropía con escala:", round(ent(pesos), 3),
      "| sin escala:", round(ent(sin_escala), 3))

## 3. `Attention` (Luong) y `AdditiveAttention` (Bahdanau) en Keras

In [ ]:
consulta = keras.Input(shape=(seq_q, d))
valores = keras.Input(shape=(seq_k, d))
attn = layers.Attention()([consulta, valores])              # dot-product (Luong)
add_attn = layers.AdditiveAttention()([consulta, valores])  # additive (Bahdanau)
print("Attention:", attn.shape, "| AdditiveAttention:", add_attn.shape)

## 4. `MultiHeadAttention` como self-attention

In [ ]:
x = keras.Input(shape=(seq_q, d))
mha = layers.MultiHeadAttention(num_heads=2, key_dim=4)
self_attn = mha(x, x)          # self-attention: query = value = x
print("self-attention:", self_attn.shape)
# Cada head aprende un patrón distinto en un subespacio de d.

## 5. Máscara causal (generación autoregresiva)

In [ ]:
causal = mha(x, x, use_causal_mask=True)   # cada posición solo mira <= t
print("self-attention causal:", causal.shape)
mask = np.tril(np.ones((seq_q, seq_q)))    # triangular inferior a mano
print("máscara causal:\n", mask.astype(int))

## 6. Cross-attention: el decoder mira al encoder + pesos

In [ ]:
q_dec = keras.Input(shape=(seq_q, d))
kv_enc = keras.Input(shape=(seq_k, d))
salida_cross, pesos_cross = layers.MultiHeadAttention(num_heads=2, key_dim=4)(
    q_dec, kv_enc, return_attention_scores=True)   # Q del decoder, K/V del encoder
print("cross-attention:", salida_cross.shape, "| pesos:", pesos_cross.shape)

## Ejercicios

1. **Attention a mano**: con Q, K, V random verificá shapes y que los pesos suman 1.
2. **Efecto del escalado**: compará la entropía de los pesos con y sin `/√d`.
3. **`MultiHeadAttention`**: aplicá `mha(x, x)` (self) y `mha(q, kv)` (cross) y compará shapes.
4. **Máscara causal**: usá `use_causal_mask=True` y verificá que no se mira el futuro.

## Conclusiones

- La atención pondera valores según la afinidad **query-key**: `softmax(QK^T/√d)·V`.
- El **escalado por √d** evita que el softmax se sature cuando `d` es grande.
- **Self-attention** (`Q=K=V`) es la base del Transformer; **cross-attention** conecta decoder→encoder.
- **Multi-head** corre varias atenciones en subespacios distintos y concatena.
- La máscara **causal** impide ver el futuro; los pesos son interpretables como alineamiento.